# BBC News: Sport Category Sub-classification
## NLP Pipeline: BGE-M3 Embeddings + BERTopic (UMAP + HDBSCAN + c-TF-IDF)

**Architecture:**
- Layer 1: Data Ingestion — Raw .txt files → List[String]
- Layer 2: BGE-M3 Embeddings — 8192 token context, 1024 dims
- Layer 3: UMAP — Non-linear reduction 1024 → 5 dims
- Layer 4: HDBSCAN — Automatic density-based clustering
- Layer 5a: CountVectorizer — Removes noise words per cluster
- Layer 5b: c-TF-IDF — Distinctive keywords across clusters
- Layer 6: KeyBERTInspired — Semantic keyword refinement
- Layer 7: Output — DataFrame [filename, category, sub_category, confidence_score, preview]

**Dataset:** BBC News Sport Category 511 raw articles
**Final Output:** DataFrame mapping every article to its sport sub-category

## Environment Setup & Imports

In [ ]:
# Core libraries
import os
import re
import pickle
import numpy as np
import pandas as pd
from loader import load_data

# Sentence Transformers — BGE-M3
from sentence_transformers import SentenceTransformer

# BERTopic pipeline
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

print("All imports successful")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

ModuleNotFoundError: No module named 'pandas'

## Data Ingestion Layer
### Loading raw business articles from local storage
### Source: BBC News dataset for business category
### No aggressive cleaning at this stage, BGE-M3 needs full sentence structure

In [ ]:
# Load all articles and labels
load_data = load_data('../data')


# Load all txt files
Sport_docs = []
Sport_files = []

for filename in sorted(os.listdir(load_data)):
    if filename.endswith('.txt'):
        file_path = os.path.join(load_data, filename)
        with open(file_path, encoding='utf-8', errors='ignore') as f:
            text = f.read()
        Sport_docs.append(text)
        Sport_files.append(filename)

        
print(f"Total Sport articles loaded: {len(Sport_docs)}")
print(f"\nSample filename: {Sport_files[0]}")
print(f"\nSample article preview:")
print(f"{Sport_docs[0][:300]}")

## Data Normalization Layer
### Light cleaning only preserving full sentence structure for BGE-M3
### We only remove: encoding errors, excessive whitespace, HTML tags
### Punctuation, proper nouns, stopwords intentionally preserved

In [ ]:
def light_clean(text):
    # Fix encoding errors
    text = text.encode('utf-8', 'ignore').decode('utf-8')

    # Remove HTML tags if present
    text = re.sub(r'<.*?>', '', text)

    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply light cleaning
cleaned_Sport = [light_clean(doc) for doc in Sport_docs]

print(f"Cleaning complete — {len(cleaned_Sport)} articles processed")
print(f"\nBefore: {Sport_docs[0][:200]}")
print(f"\nAfter: {cleaned_Sport[0][:200]}")

##  Deduplication
### Removing duplicate articles before embedding generation
### Identical documents produce identical vectors hurts clustering quality
### Preserving filename mapping for final DataFrame output

In [ ]:
# Remove duplicates while preserving filename mapping
seen = set()
unique_Sport = []
unique_files = []

for doc, filename in zip(cleaned_Sport, Sport_files):
    if doc not in seen:
        seen.add(doc)
        unique_Sport.append(doc)
        unique_files.append(filename)

print(f"Before deduplication: {len(cleaned_Sport)} articles")
print(f"After deduplication:  {len(unique_Sport)} articles")
print(f"Duplicates removed:   {len(cleaned_Sport) - len(unique_Sport)}")

## Embedding Layer (BGE-M3)
### Model: BAAI/bge-m3 —> 8192 token context window, 1024 dimensional output
### Full-sequence self-attention transformer from BAAI
### Every article processed as one complete unit, no truncation
### Produces dense semantic vectors capturing meaning, context and intent

In [ ]:
# Load BGE-M3 model
print("Loading BGE-M3 model...")
model = SentenceTransformer('BAAI/bge-m3')
print("BGE-M3 loaded successfully")
print(f"Max sequence length: {model.max_seq_length}")

##  Generate BGE-M3 Document Embeddings
### Each article is encoded as a single 1024-dimensional semantic vector
### BGE-M3 reads the entire document using full-sequence self-attention
### batch_size=32 processes 32 articles at a time to manage memory efficiently
### show_progress_bar=True lets us track encoding progress

In [ ]:
print("Generating BGE-M3 embeddings for Sport articles...")
print("This will take a few minutes...")

embeddings = model.encode(
    unique_Sport,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"\nEmbeddings generated successfully")
print(f"Shape: {embeddings.shape}")
print(f"Each article \u2192 {embeddings.shape[1]} dimensional vector")

## BERTopic Pipeline (All 6 Modules)
### Module 3: UMAP — compresses 1024 dims to 5, random_state=42 for stability
### Module 4: HDBSCAN — finds natural sub-categories automatically
### Module 5a: CountVectorizer — removes noise words before keyword extraction
### Module 5b: c-TF-IDF — finds distinctive keywords per cluster
### Module 6: KeyBERTInspired — refines keywords using semantic similarity

In [ ]:
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN


# Module 3 — UMAP
umap_model = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.01,
    random_state=42
)

# Module 4 — HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=10,  # Adjusted to identify more, smaller clusters
    min_samples=3,        # Maintained to allow for more fine-grained clusters
    metric='euclidean',
    cluster_selection_method='eom'
)

# Module 5a — CountVectorizer
vectorizer_model = CountVectorizer(
    stop_words="english",
    max_df=0.8,
    ngram_range=(1, 2)
)

# Module 5b — c-TF-IDF
ctfidf_model = ClassTfidfTransformer(
    bm25_weighting=True
)

# Module 6 — KeyBERTInspired
representation_model = KeyBERTInspired()

# Assemble full BERTopic pipeline
topic_model = BERTopic(
    embedding_model=model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    verbose=True
)

print("BERTopic pipeline assembled successfully")
print("All 6 modules configured")

## Fit BERTopic to BGE-M3 Embeddings
### Passing pre-computed BGE-M3 embeddings directly into BERTopic
### BERTopic runs all 6 modules sequentially:
### UMAP → HDBSCAN → CountVectorizer → c-TF-IDF → KeyBERTInspired
### topics_ contains sub-category assignment for every article
### probs_ contains confidence score for each assignment

In [ ]:
# Fit BERTopic using pre-computed BGE-M3 embeddings
print("Running BERTopic pipeline...")
print("UMAP \u2192 HDBSCAN \u2192 Vectorizer \u2192 c-TF-IDF \u2192 KeyBERT")
print("This may take a few minutes...")

topics, probs = topic_model.fit_transform(
    unique_Sport,
    embeddings=embeddings
)

print(f"\nBERTopic complete")
print(f"Total articles processed: {len(topics)}")
print(f"Sub-categories discovered: {len(set(topics)) - 1}")
print(f"Noise articles (topic -1): {topics.count(-1)}")

## view subcategories from keyBERTHinsipired

In [ ]:
# View all discovered sub-categories and their keywords
topic_info = topic_model.get_topic_info()
print("All discovered sub-categories:")
print("=" * 60)
for _, row in topic_info.iterrows():
    if row['Topic'] != -1:
        print(f"\nTopic {row['Topic']}:")
        print(f"  Auto-label: {row['Name']}")
        print(f"  Articles: {row['Count']}")

## Output & Evaluation Layer

### Visualization: Intertopic Distance Map

This visualization helps us understand the relationships between the discovered sub-categories (topics). Topics that are closer together in the 2D plane are more semantically similar, while those further apart are more distinct. The size of the circle represents the frequency of the topic.


In [ ]:
import plotly.express as px

# Visualize topics
fig = topic_model.visualize_topics()

# Ensure the figure is displayable in Colab
fig.show()


## POST-CLUSTERING FIX


In [ ]:

# POST-CLUSTERING FIX 
#
# HDBSCAN merged international football and rugby into one cluster
# because both use squad/injury/international vocabulary.
# We fix this by inspecting the article text directly.

# Keywords that only appear in rugby articles
RUGBY_SIGNALS = [
    "fly-half", "scrum-half", "flanker", "lock", "prop", "hooker",
    "number eight", "fullback", "line-out", "scrum", "ruck", "maul",
    "twickenham", "six nations", "heineken cup", "lions", "rugby union",
    "rugby league", "drop goal", "conversion", "tries", "sin bin",
    "irb", "rbs", "premiership rugby"
]

# Keywords that only appear in football articles
FOOTBALL_SIGNALS = [
    "premier league", "fa cup", "champions league", "la liga",
    "bundesliga", "serie a", "world cup qualifier", "uefa",
    "striker", "midfielder", "goalkeeper", "winger", "defender",
    "transfer fee", "loan deal", "penalty shootout", "hat-trick",
    "yellow card", "red card", "offside", "corner kick"
]

def split_topic_1(topic_id, text):
    """
    For articles in Topic 1 only, inspect the text and reclassify
    as rugby (1) or international football (6) based on signal keywords.
    Articles with no clear signal stay in Topic 1 (rugby) as default.
    """
    if topic_id != 1:
        return topic_id  # leave all other topics untouched

    text_lower = text.lower()

    rugby_score    = sum(1 for kw in RUGBY_SIGNALS    if kw in text_lower)
    football_score = sum(1 for kw in FOOTBALL_SIGNALS if kw in text_lower)

    # Football wins if it scores higher — otherwise stays as rugby
    if football_score > rugby_score:
        return 6   # new topic ID for International Football
    return 1       # stays as Rugby Union

# Apply the split to the topics list before building final_df
corrected_topics = [
    split_topic_1(t, text)
    for t, text in zip(topics, unique_Sport)
]

## Final Sub-category DataFrame
### Mapping all 503 Sport articles to their discovered sub-categories
### Columns: filename, category, sub_category, confidence_score, preview

In [ ]:
#  FINAL DATAFRAME WITH CONFIDENCE SCORES & NOISE RELABELLING
# Sport consolidation
sport_consolidation = {
    5: 4,
}

# Human-readable label mapping
topic_labels = {
    0:  "Football Match Reports",
    1:  "Rugby Union",
    2:  "Tennis Tournaments",
    3:  "Athletics Track Field",
    4:  "Athletics Doping",
    6:  "International Football",   # new — split from Topic 1
    -1: "Unassigned"
}

# Noise remap: 3 noise articles
noise_remap = {
    "046.txt": 4,    # Radcliffe drugs stance → Athletics Doping
    "397.txt": 1,    # Sculthorpe Lions captaincy → Rugby Union
    "398.txt": 1,    # Tigers/Farrell code switch → Rugby Union
}

# Manual remap: 6 misclassified articles in Topic 1
# These scored zero rugby signals they are football articles that
# HDBSCAN placed in the rugby cluster due to shared squad/injury vocab.
manual_remap = {
    "123.txt": 6,    
    "131.txt": 6,    
    "133.txt": 6,    
    "128.txt": 6,    
    "212.txt": 6,    
    "136.txt": 6,
    "113.txt": 6,
    "114.txt": 0,
    "119.txt": 6,
    "127.txt": 0,
    "129.txt": 0,
    "132.txt": 6,
    "134.txt": 6,
    "293.txt": 1,

}

# Build initial DataFrame
final_df = pd.DataFrame({
    'filename':         unique_files,
    'category':         'Sport',
    'sub_category_id':  topics,
    'sub_category':     [topic_labels.get(t, 'Unassigned') for t in topics],
    'confidence_score': [round(float(p), 4) if p > 0 else 0.0 for p in probs],
    'preview':          unique_Sport
})

# Apply all remapping in one pass
def consolidate_and_remap(row):
    """
    Three jobs in one pass — applied in priority order:
    1. Noise articles (-1) → assign correct sport via noise_remap
    2. Misclassified articles → correct via manual_remap
    3. Fragmented topics → consolidate via sport_consolidation
    """
    sid = row['sub_category_id']

    # Step 1 — handle noise
    if sid == -1:
        sid = noise_remap.get(row['filename'], -1)

    # Step 2 — fix known misclassified articles
    if row['filename'] in manual_remap:
        sid = manual_remap[row['filename']]

    # Step 3 — consolidate fragmented topics
    sid = sport_consolidation.get(sid, sid)

    return sid, topic_labels.get(sid, 'Unassigned')

final_df[['sub_category_id', 'sub_category']] = final_df.apply(
    consolidate_and_remap, axis=1, result_type='expand'
)

# Summary table
print("\nBBC SPORT — SUB-CATEGORY SUMMARY (fully labelled)")
print("=" * 65)

summary = final_df.groupby(['sub_category_id', 'sub_category']).agg(
    article_count=('filename', 'count'),
    avg_confidence=('confidence_score', 'mean')
).reset_index()

summary['avg_confidence'] = summary['avg_confidence'].round(4)
summary = summary.sort_values('sub_category_id')

for _, row in summary.iterrows():
    print(f"\n  [{int(row['sub_category_id'])}] {row['sub_category']}")
    print(f"       Articles: {row['article_count']} | Avg Confidence: {row['avg_confidence']}")

print(f"\n{'=' * 65}")
print(f"Total articles    : {len(final_df)}")
print(f"Sub-categories    : {final_df['sub_category_id'].nunique()}")
print(f"Unassigned (noise): {(final_df['sub_category_id'] == -1).sum()}")

# Full DataFrame display
print("\nFull DataFrame:")
display(final_df[['filename', 'category', 'sub_category', 'confidence_score', 'preview']])